In [1]:
import pickle
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

data = pd.read_csv("/Users/user/Desktop/CreditScoreClassification/train.csv")

def convert_to_months(df):
    df = df.copy()
    def convert(x):
        try:
            # Attempt to convert 'x' to months
            return int(x.split(' ')[0]) * 12 + int(x.split(' ')[3])
        except (AttributeError, ValueError):
            # If error occurs, return NaN
            return np.nan
    df['Credit_History_Age_in_Months'] = df['Credit_History_Age'].apply(convert)
    return df

# Convert 'Credit_History_Age' to months
data = convert_to_months(data)

# Numeric and categorical feature lists
numeric_features = [
    'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 
    'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 
    'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 
    'Credit_Utilization_Ratio', 'Total_EMI_per_month', 
    'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_in_Months',
    'Age'  
]

def clean_numeric_column(df, column_name):
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    return df

for col in numeric_features:
    data = clean_numeric_column(data, col)


categorical_features = [ 'Occupation', 'Type_of_Loan'] 


X = data[numeric_features + [ 'Occupation', 'Type_of_Loan']]
y = data['Credit_Score']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing pipeline for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # Impute missing values with the median
            ('scaler', StandardScaler())  # Scale numeric features
        ]), numeric_features),
        
        ('cat', Pipeline([
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))  # OneHotEncoding for categorical features
        ]), categorical_features)
    ]
)

# Full pipeline with CatBoost classifier
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', CatBoostClassifier(iterations=500, depth=6, learning_rate=0.1, verbose=0))
])

# Train the pipeline
pipeline.fit(X_train, y_train)

# Get prediction probabilities on the test set
y_pred_proba = pipeline.predict_proba(X_test)

# Save prediction probabilities to a CSV file (optional)
proba_df = pd.DataFrame(y_pred_proba, columns=pipeline.named_steps['classifier'].classes_)
proba_df.to_csv("prediction_probabilities.csv", index=False)

print("✅ Prediction probabilities saved successfully!")

# Save the trained pipeline
with open("catboost_model_pipeline.pkl", "wb") as model_file:
    pickle.dump(pipeline, model_file)

print("✅ Pipeline (with preprocessing and model) saved successfully!")



/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/var/folders/5z/jxsqfzd56z75p0hz3j1vd5v40000gn/T/ipykernel_1150/2340710540.py:11: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("/Users/user/Desktop/CreditScoreClassification/train.csv")


✅ Prediction probabilities saved successfully!
✅ Pipeline (with preprocessing and model) saved successfully!


In [2]:
# import pandas as pd
# pd.read_csv("/Users/user/Desktop/CreditScoreClassification/prediction_probabilities.csv")

In [3]:
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the saved pipeline
with open("catboost_model_pipeline.pkl", "rb") as model_file:
    loaded_pipeline = pickle.load(model_file)

print("✅ Model loaded successfully!")

# Load new test dataset
test_data = pd.read_csv("/Users/user/Desktop/CreditScoreClassification/train.csv")

# Function to convert Credit_History_Age to months
def convert_to_months(df):
    def convert(x):
        try:
            parts = x.split(' ')
            years = int(parts[0]) * 12
            months = int(parts[2]) if len(parts) > 2 else 0
            return years + months
        except (AttributeError, ValueError, IndexError):
            return np.nan
    df['Credit_History_Age_in_Months'] = df['Credit_History_Age'].apply(convert)
    return df

# Apply transformation
test_data = convert_to_months(test_data)

# Define feature names (same as training)
numeric_features = [
    'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 
    'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 
    'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 
    'Credit_Utilization_Ratio', 'Total_EMI_per_month', 
    'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_in_Months',
    'Age'  
]
categorical_features = ['Occupation', 'Type_of_Loan']

# Convert numeric columns
def clean_numeric_column(df, column_name):
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    return df

for col in numeric_features:
    test_data = clean_numeric_column(test_data, col)

# Select features (drop target column if present)
X_test = test_data[numeric_features + categorical_features]

# Make predictions
y_test_pred_proba = loaded_pipeline.predict_proba(X_test)

# Convert to DataFrame
class_labels = loaded_pipeline.named_steps['classifier'].classes_  # Get class labels
proba_df = pd.DataFrame(y_test_pred_proba, columns=class_labels)

# Save predictions
proba_df.to_csv("test_predictions.csv", index=False)
print("✅ Test predictions saved successfully!")

# If you want to get class predictions (not probabilities), use:
y_test_pred = loaded_pipeline.predict(X_test)

# Save class predictions
pd.DataFrame(y_test_pred, columns=["Predicted_Credit_Score"]).to_csv("test_class_predictions.csv", index=False)
print("✅ Class predictions saved successfully!")


✅ Model loaded successfully!
✅ Test predictions saved successfully!
✅ Class predictions saved successfully!


In [5]:
 pd.read_csv("test_predictions.csv")

,Good,Poor,Standard
0,0.490259,0.109803,0.399939
1,0.467005,0.066479,0.466516
2,0.493983,0.089892,0.416124
3,0.586086,0.077511,0.336403
4,0.552003,0.078660,0.369337
...,...,...,...
49995,0.016448,0.506933,0.476619
49996,0.092417,0.119449,0.788134
49997,0.070714,0.260087,0.669200
49998,0.090868,0.121206,0.787926


In [6]:
pd.read_csv("test_class_predictions.csv")

,Predicted_Credit_Score
0,Good
1,Good
2,Good
3,Good
4,Good
...,...
49995,Poor
49996,Standard
49997,Standard
49998,Standard


In [ ]:

# model_path = "/Users/user/Desktop/CreditScoreClassification/catboost_model_pipeline.pkl"
# with open(model_path, "rb") as model_file:
#     pipeline = pickle.load(model_file)

# numeric_features = [
#     'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 
#     'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 
#     'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 
#     'Credit_Utilization_Ratio', 'Total_EMI_per_month', 
#     'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_in_Months',
#     'Age'
# ]

# categorical_features = ['Occupation', 'Type_of_Loan']

# st.title("Credit Score Prediction App")
# st.write("Enter the required details to predict the credit score category.")

# user_input = {}
# for col in numeric_features:
#     user_input[col] = st.number_input(f"{col} (Numeric)", value=0.0)

# for col in categorical_features:
#     user_input[col] = st.text_input(f"{col} (Categorical)", "")

# if st.button("Predict Credit Score"):
#     input_df = pd.DataFrame([user_input])

#     prediction_proba = pipeline.predict_proba(input_df)

#     class_labels = pipeline.named_steps['classifier'].classes_

#     st.write("### Prediction Probabilities:")
#     for label, prob in zip(class_labels, prediction_proba[0]):
#         st.write(f"**{label}:** {prob:.2%}")

#     predicted_class = class_labels[np.argmax(prediction_proba)]
#     st.success(f"Predicted Credit Score: **{predicted_class}**")
